In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)
print(train.head())

Training data shape: (283085, 19)
Test data shape: (177063, 19)
         date        high         low  momentum_index  beta_indicator  \
0  2012-06-22   24.797176  162.793195      153.349972       90.290064   
1  2013-02-27   84.148858   62.911473      123.066280       99.689780   
2  2010-07-19  162.714072    7.813419       50.140068      -30.373609   
3  2015-12-31 -128.017977  -38.679447      -21.881272     -173.386894   
4  2012-11-12  204.768157   -6.981525      -38.978256       64.867265   

   risk_premium     index  volatility_factor  technical_score  \
0    277.383362   62235.0         105.868473       128.280484   
1    116.635811   48784.0         116.700703         8.730951   
2    145.254746  233118.0         302.752221       -40.118103   
3   -123.314003   28867.0          -3.662459        81.801806   
4    -90.095079  182732.0        -157.223822       -76.111066   

   oscillator_value  liquidity_ratio        open  quant_index  trend_strength  \
0         38.359969      

In [2]:
# Remove training rows without a known target
train = train.dropna(subset=["close"]).copy()

y = train["close"]
X = train.drop(columns=["close"]).copy()

# Keep test labels separately for evaluation
y_test = test["close"].to_numpy(dtype=float)

# Features used for prediction
X_test = test.drop(columns=["close"]).copy()

print("Features:", X.shape)
print("Target:", y.shape)
print("Test features:", X_test.shape)

Features: (281575, 18)
Target: (281575,)
Test features: (177063, 18)


In [3]:
numeric_cols = X.select_dtypes(include=np.number).columns
text_cols = X.select_dtypes(exclude=np.number).columns

# Use training-data medians to fill missing numeric values
medians = X[numeric_cols].median()

X[numeric_cols] = X[numeric_cols].fillna(medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(medians)

# Fill missing categorical values
X[text_cols] = X[text_cols].fillna("Unknown")
X_test[text_cols] = X_test[text_cols].fillna("Unknown")

# Convert categorical columns into numeric indicator columns
X = pd.get_dummies(X, columns=text_cols)
X_test = pd.get_dummies(X_test, columns=text_cols)

# Make training and test columns match
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print("Training features after encoding:", X.shape)
print("Test features after encoding:", X_test.shape)

Training features after encoding: (281575, 1784)
Test features after encoding: (177063, 1784)


In [4]:
X = X.to_numpy(dtype=float)
X_test = X_test.to_numpy(dtype=float)
y = y.to_numpy(dtype=float)

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)


X: (281575, 1784)
y: (281575,)
X_test: (177063, 1784)


In [5]:
np.random.seed(42)
indices = np.random.permutation(len(X)).astype(np.int32)

split = int(0.8 * len(X))

train_idx = indices[:split]
val_idx = indices[split:]

X_train = X[train_idx]
y_train = y[train_idx]

X_val = X[val_idx]
y_val = y[val_idx]
del indices, train_idx, val_idx

print("Training split:", X_train.shape)
print("Validation split:", X_val.shape)

Training split: (225260, 1784)
Validation split: (56315, 1784)


In [6]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

std[std == 0] = 1

X_train_scaled = (X_train - mean) / std
X_val_scaled = (X_val - mean) / std

X_test_scaled = (X_test - mean) / std

print("Features standardized!")

Features standardized!


In [7]:
X_train_b = np.column_stack((
    np.ones(len(X_train_scaled)),
    X_train_scaled
))

X_val_b = np.column_stack((
    np.ones(len(X_val_scaled)),
    X_val_scaled
))

X_test_b = np.column_stack((
    np.ones(len(X_test_scaled)),
    X_test_scaled
))

print("Number of weights:", X_train_b.shape[1])

Number of weights: 1785


In [8]:
weights = np.zeros(X_train_b.shape[1])

learning_rate = 0.01
epochs = 1000

n = len(y_train)

train_costs = []
val_costs = []

print("Initial weights:", len(weights))
print("Epochs:", epochs)

Initial weights: 1785
Epochs: 1000


In [ ]:
for epoch in range(epochs):

    predictions = X_train_b @ weights

    errors = predictions - y_train

    gradient = (2 / n) * (X_train_b.T @ errors)

    weights = weights - learning_rate * gradient

    train_predictions = X_train_b @ weights
    train_mse = np.mean((train_predictions - y_train) ** 2)

    val_predictions = X_val_b @ weights
    val_mse = np.mean((val_predictions - y_val) ** 2)

    train_costs.append(train_mse)
    val_costs.append(val_mse)

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1}: "
            f"Training MSE = {train_mse:.4f}, "
            f"Validation MSE = {val_mse:.4f}"
        )

print("Training complete!")
print("Total epochs recorded:", len(train_costs))

Epoch 50: Training MSE = 1127.4202, Validation MSE = 1163.1889
Epoch 100: Training MSE = 533.9569, Validation MSE = 565.3848
Epoch 150: Training MSE = 398.2094, Validation MSE = 427.3759
Epoch 200: Training MSE = 335.0984, Validation MSE = 362.5271
Epoch 250: Training MSE = 290.7205, Validation MSE = 316.7424
Epoch 300: Training MSE = 255.8957, Validation MSE = 280.7786
Epoch 350: Training MSE = 227.9460, Validation MSE = 251.9132
Epoch 400: Training MSE = 205.3999, Validation MSE = 228.6350
Epoch 450: Training MSE = 187.1834, Validation MSE = 209.8361
Epoch 500: Training MSE = 172.4539, Validation MSE = 194.6449
Epoch 550: Training MSE = 160.5380, Validation MSE = 182.3642
Epoch 600: Training MSE = 150.8947, Validation MSE = 172.4337
Epoch 650: Training MSE = 143.0882, Validation MSE = 164.4018
Epoch 700: Training MSE = 136.7673, Validation MSE = 157.9047
Epoch 750: Training MSE = 131.6481, Validation MSE = 152.6483
Epoch 800: Training MSE = 127.5014, Validation MSE = 148.3955
Epoch 8

In [ ]:
epoch_results = pd.DataFrame({
    "Epoch": range(1, len(train_costs) + 1),
    "Training MSE": train_costs,
    "Validation MSE": val_costs
})

print(epoch_results.head())
print(epoch_results.tail())
print("Total recorded epochs:", len(epoch_results))

In [ ]:
epoch_results.to_csv("epoch_results.csv", index=False)

print("Saved epoch_results.csv")

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    epoch_results["Epoch"],
    epoch_results["Training MSE"],
    label="Training MSE"
)

plt.plot(
    epoch_results["Epoch"],
    epoch_results["Validation MSE"],
    label="Validation MSE"
)

plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error (MSE)")
plt.title("Linear Regression: Cost vs Epochs")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Predict close values for every test row
test_predictions = X_test_b @ weights

# Identify rows with known actual close values
valid_rows = np.isfinite(y_test)

actual = y_test[valid_rows]
predicted = test_predictions[valid_rows]

# Calculate evaluation metrics
test_mse = np.mean((actual - predicted) ** 2)
test_rmse = np.sqrt(test_mse)
test_mae = np.mean(np.abs(actual - predicted))

print(f"Test MSE:  {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")

print("Rows evaluated:", valid_rows.sum())
print("Rows with missing labels:", (~valid_rows).sum())

In [ ]:
predictions_df = pd.DataFrame({
    "predicted_close": test_predictions
})

predictions_df.to_csv("predictions.csv", index=False)

print("Test rows:", len(test))
print("Predictions generated:", len(test_predictions))
print("Rows saved:", len(predictions_df))
print(
    "Missing predictions:",
    predictions_df["predicted_close"].isna().sum()
)